In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
# Define severity class colors (matching R code)
SEVERITY_COLORS = {
    "Unburned to Low": "#006400",
    "Low": "#7fffd4",
    "Moderate": "#ffff00",
    "High": "#ff0000"
}

# Define access class mapping and order
ACCESS_MAPPING = {
    "roaded": "Developed",
    "roadless": "IRA",
    "wilderness": "Wilderness"
}
ACCESS_ORDER = ["Developed", "IRA", "Wilderness"]


In [ ]:
def load_and_prepare_data(
    mtbs_path: str, 
    extent_path: str, 
    start_year: int = 2014, 
    end_year: int = 2023
    ) -> pd.DataFrame:
    """
    Load and prepare data for severity analysis.
    
    Args:
        mtbs_path: Path to MTBS summary CSV
        extent_path: Path to extent CSV
        start_year: Start year for analysis
        end_year: End year for analysis
    
    Returns:
        DataFrame with burned area and proportion by access class and severity
    """
    # Load data
    mtbs_summary = pd.read_csv(mtbs_path)
    area_data = pd.read_csv(extent_path)
    
    # Standardize column names
    if 'access' in mtbs_summary.columns:
        mtbs_summary = mtbs_summary.rename(columns={'access': 'access_class'})
    if 'access' in area_data.columns:
        area_data = area_data.rename(columns={'access': 'access_class'})
    
    # Clean MTBS data
    mtbs_clean = mtbs_summary.copy()
    mtbs_clean['mtbs_class'] = pd.to_numeric(mtbs_clean['mtbs_class'], errors='coerce')
    mtbs_clean = mtbs_clean[mtbs_clean['mtbs_class'].isin([1, 2, 3, 4])].copy()
    
    # Map severity classes to labels
    severity_map = {
        1: "Unburned to Low",
        2: "Low",
        3: "Moderate",
        4: "High"
    }
    mtbs_clean['mtbs_class'] = mtbs_clean['mtbs_class'].map(severity_map)
    
    # Clean area data
    area_clean = area_data.copy()
    area_clean['extent_masked_km2'] = pd.to_numeric(area_clean['extent_masked_km2'], errors='coerce')
    area_clean['access_class'] = area_clean['access_class'].str.lower().str.strip()
    
    # Calculate total extent by access class (all states combined)
    extent_allstates = area_clean.groupby('access_class', as_index=False)['extent_masked_km2'].sum()
    extent_allstates = extent_allstates.rename(columns={'extent_masked_km2': 'extent_masked_km2_all'})
    
    # Filter to year range and summarize
    burned_summary = mtbs_clean[(mtbs_clean['year'] >= start_year) & 
                                (mtbs_clean['year'] <= end_year)].copy()
    
    burned_agg = burned_summary.groupby(['access_class', 'mtbs_class'], as_index=False)['area_km2'].sum()
    burned_agg = burned_agg.rename(columns={'area_km2': 'area_burned_km2'})
    
    # Join with extent data
    burned_with_extent = burned_agg.merge(extent_allstates, on='access_class', how='left')
    
    # Calculate proportion burned
    burned_with_extent['prop_burned'] = (burned_with_extent['area_burned_km2'] / 
                                         burned_with_extent['extent_masked_km2_all'])
    
    # Recode access class labels
    burned_with_extent['access_class'] = burned_with_extent['access_class'].map(ACCESS_MAPPING)
    
    # Set categorical order for access class
    burned_with_extent['access_class'] = pd.Categorical(
        burned_with_extent['access_class'],
        categories=ACCESS_ORDER,
        ordered=True
    )
    
    # Set categorical order for severity
    burned_with_extent['mtbs_class'] = pd.Categorical(
        burned_with_extent['mtbs_class'],
        categories=["Unburned to Low", "Low", "Moderate", "High"],
        ordered=True
    )
    
    return burned_with_extent


In [ ]:
def create_stacked_barplot(ax, data, title, show_ylabel=True, show_legend=False):
    """
    Create a stacked bar plot for severity by access class.
    
    Args:
        ax: Matplotlib axis object
        data: DataFrame with columns: access_class, mtbs_class, prop_burned
        title: Plot title
        show_ylabel: Whether to show y-axis label
        show_legend: Whether to show legend
    """
    # Pivot data for stacked bar plot
    pivot_data = data.pivot(index='access_class', columns='mtbs_class', values='prop_burned')
    pivot_data = pivot_data.fillna(0)
    
    # Ensure all severity classes are present in correct order
    severity_order = ["Unburned to Low", "Low", "Moderate", "High"]
    for sev in severity_order:
        if sev not in pivot_data.columns:
            pivot_data[sev] = 0
    pivot_data = pivot_data[severity_order]
    
    # Create stacked bar plot
    x_pos = np.arange(len(pivot_data.index))
    bottom = np.zeros(len(pivot_data.index))
    
    bars = []
    for severity in severity_order:
        if severity in pivot_data.columns:
            bar = ax.bar(x_pos, pivot_data[severity], bottom=bottom,
                        color=SEVERITY_COLORS[severity], label=severity,
                        width=0.7, edgecolor='black', linewidth=0.8)
            bars.append(bar)
            bottom += pivot_data[severity].values
    
    # Formatting
    ax.set_xticks(x_pos)
    ax.set_xticklabels(pivot_data.index, fontsize=14)
    ax.set_xlabel("", fontsize=16)
    
    if show_ylabel:
        ax.set_ylabel("Proportion of Access Class Burned", fontsize=16)
    
    ax.set_title(title, fontsize=16)
    ax.set_ylim(0, bottom.max() * 1.02)
    
    # Format y-axis as percentage
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    ax.tick_params(axis='y', labelsize=14)
    
    # Remove vertical gridlines, keep horizontal
    ax.grid(axis='y', linestyle=':', alpha=0.5)
    ax.set_axisbelow(True)
    
    if show_legend:
        ax.legend(title="MTBS Severity", loc="upper right", frameon=False, fontsize=14, title_fontsize=14)


In [ ]:
def plot_severity_comparison(
    data_full: pd.DataFrame, 
    data_recent: pd.DataFrame, 
    out_dir: str
    ) -> str:
    """
    Create side-by-side severity plots: cumulative (full period) vs recent period (2014-2023).
    
    Args:
        data_full: DataFrame with burned area data for full time period
        data_recent: DataFrame with burned area data for 2014-2023
        out_dir: Output directory for plot
    
    Returns:
        Path to saved plot
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6), constrained_layout=True)
    
    # Left plot: Cumulative proportion burned (full time period)
    create_stacked_barplot(ax1, data_full, "Cumulative Proportion Burned (1984-2023)", 
                          show_ylabel=True, show_legend=False)
    
    # Right plot: Recent period (2014-2023)
    create_stacked_barplot(ax2, data_recent, "Proportion Burned (2014-2023)", 
                          show_ylabel=False, show_legend=True)
    
    # Add panel labels
    ax1.text(0.02, 0.98, 'C', transform=ax1.transAxes, fontsize=48, 
             fontweight='bold', va='top', ha='left')
    ax2.text(0.02, 0.98, 'D', transform=ax2.transAxes, fontsize=48, 
             fontweight='bold', va='top', ha='left')
    
    # Ensure both plots have same y-axis scale
    y_max = max(ax1.get_ylim()[1], ax2.get_ylim()[1])
    ax1.set_ylim(0, y_max)
    ax2.set_ylim(0, y_max)
    
    # Save figure
    os.makedirs(out_dir, exist_ok=True)
    fname = "mtbs_severity_barplot_comparison.png"
    fpath = os.path.join(out_dir, fname)
    plt.savefig(fpath, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    
    return fpath


In [ ]:
def create_severity_barplot(
    mtbs_path: str, 
    extent_path: str, 
    out_dir: str
    ) -> str:
    """
    Main function to create severity bar plot comparison.
    
    Args:
        mtbs_path: Path to MTBS summary CSV
        extent_path: Path to extent CSV
        out_dir: Output directory
    
    Returns:
        Path to saved plot
    """
    # Prepare data for full time period (cumulative)
    data_full = load_and_prepare_data(mtbs_path, extent_path, 
                                      start_year=1984, end_year=2023)
    
    # Prepare data for recent period (2014-2023)
    data_recent = load_and_prepare_data(mtbs_path, extent_path, 
                                        start_year=2014, end_year=2023)
    
    return plot_severity_comparison(data_full, data_recent, out_dir)


## File I/O and logic

In [ ]:
# Define a path to the project folder
project_folder = "<PATH/TO/PROJECT/FOLDER>"

# Generate the plot
path = create_severity_barplot(
    mtbs_path=os.path.join(data_folder, "data", "tables", "mtbs_nfs_access_class_summary.csv"),
    extent_path=os.path.join(data_folder, "data", "tables", "nfs_access_extent_km2.csv"),
    out_dir=os.path.join(data_folder, "figures")
)
print(f"Created plot: {path}")